# Questions about building the AR4 arm

A person building the AR4 robot arm works from a 377-page manual. At each assembly step they look at a bench photo and its printed instruction and ask what the step needs: how many screws and which kind, which part goes in, which tool. On the parts-list pages they ask for facts, such as how many of one screw the whole robot needs. Each question is answered by reading an image and a sentence together, which is the input a vision-language model (VLM) takes.

In this assignment you measure how well [SmolVLM-256M-Instruct](https://arxiv.org/abs/2504.05299) answers these questions without training. You then fine-tune it in three ways and compare them. [LoRA](https://arxiv.org/abs/2106.09685) adds trainable low-rank matrices beside frozen attention and MLP weights. [DoRA](https://arxiv.org/abs/2402.09353) splits each of those weights into a magnitude and a direction and applies the low-rank update to the direction only. [LayerNorm tuning](https://arxiv.org/abs/2312.11420) trains nothing but the scale of the normalization layers. The first two train about three million parameters, the third about thirty-five thousand, out of 256 million.

In [ ]:
%pip install -q -U "transformers>=5" peft pymupdf datasets

In [ ]:
import hashlib
import inspect
import json
import marshal
import re
import time
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pymupdf
import torch
from datasets import load_dataset
from PIL import Image
from transformers import AutoModelForImageTextToText, AutoProcessor
from transformers.utils import logging as hf_logging

hf_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message="Unsupported layer type")  # RMSNorm under LayerNorm tuning

## Settings

Keep these values for the runs you hand in. `TRAIN_LIMIT` and `TEST_LIMIT` are there so that you can check your code on a few pairs first.

In [ ]:
MODEL_ID = "HuggingFaceTB/SmolVLM-256M-Instruct"
DATASET_ID = "aegean-ai/ar4-assembly-vqa"
MANUAL_URL = "https://raw.githubusercontent.com/Annin-Robotics/downloads/main/2026/02/AR4-MK4-Robot-Manul-1.1.pdf"
MANUAL_SHA256 = "44d5d8d60a94078f28744d449008dd5a80ef52ce46a400607ebb02d3eb8a7fd1"
EPOCHS = 2  # passes over the training pairs, every method
LR = 2e-4  # AdamW learning rate, every method
ACCUM = 4  # pairs per optimizer step
IMAGE_EDGE = 512  # longest image side kept by the processor, in pixels
TRAIN_LIMIT = None  # None trains on every pair
TEST_LIMIT = None  # None scores every test pair
SETTINGS = {"dataset": DATASET_ID, "epochs": EPOCHS, "lr": LR, "accum": ACCUM, "image_edge": IMAGE_EDGE,
            "train_limit": TRAIN_LIMIT, "test_limit": TEST_LIMIT}  # a saved run is reused only under these and unchanged method code
RUNS = Path("runs")  # one CSV and one JSON per method
DEVICE = "cuda"
RUNS.mkdir(exist_ok=True)

## The manual and its images

Each row of the dataset names a page of the manual and a box on that page, in PDF points (1/72 inch). The image for a question is that box, rendered at 110 dots per inch. A step box covers a whole panel: the bench photo on the left and the printed instruction on the right. The answer to a step question is therefore printed inside its image, and the model has to find it there and tie it to what the photo shows. Documentation boxes cover a grid of labeled part photos or the fastener table.

The boxes were measured on version 1.1 of the manual. A later revision moves things around on its pages, so the download is checked against the SHA-256 digest of version 1.1 before any box is cut.

In [ ]:
MANUAL = Path("AR4-MK4-Robot-Manul-1.1.pdf")
if not MANUAL.exists():
    urllib.request.urlretrieve(MANUAL_URL, MANUAL)
digest = hashlib.sha256(MANUAL.read_bytes()).hexdigest()
assert digest == MANUAL_SHA256, f"{MANUAL} is not version 1.1 of the manual (sha256 {digest})"
manual = pymupdf.open(MANUAL)
_images = {}


def render(page, bbox):
    # The image a dataset row refers to: its box on its page, at 110 dpi.
    key = (int(page), tuple(float(v) for v in bbox))
    if key not in _images:
        pix = manual[key[0] - 1].get_pixmap(clip=pymupdf.Rect(*key[1]), dpi=110)
        _images[key] = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)
    return _images[key]


print(f"{manual.page_count} pages, digest matches version 1.1")

## The question-answer pairs

Step questions (`kind` is `step`) come from the assembly chapter, pages 51 to 241. They ask for a fastener count or type, the part installed, a tool or material, a specification such as a wire length, an action, or the joint a step belongs to. Documentation questions (`kind` is `doc`) come from the parts list, pages 16 to 50. They ask what a photo in a grid is called, where a part sits in the grid, or how many of a fastener the whole robot needs.

Consecutive assembly steps reuse nearly the same photo, so a random split would put near-copies of test images into training. The split is by page instead. The assembly pages are cut into blocks of ten and every fifth block is held out for test. Training pages right beside a test block are dropped, and so are training pairs whose panel contains a photo that also appears in a test panel.

In [ ]:
data = load_dataset(DATASET_ID)
train_df = data["train"].to_pandas()
test_df = data["test"].to_pandas()
if TRAIN_LIMIT:
    train_df = train_df.sample(TRAIN_LIMIT, random_state=0)
if TEST_LIMIT:
    test_df = test_df.sample(TEST_LIMIT, random_state=0)
print(f"{len(train_df)} training pairs, {len(test_df)} test pairs")
pairs = pd.concat([train_df.assign(split="train"), test_df.assign(split="test")])
pd.crosstab(pairs.qtype, [pairs.split, pairs.kind])

In [ ]:
full_test = data["test"].to_pandas()
examples = [full_test[full_test.kind == "step"].iloc[0], full_test[full_test.qtype == "where"].iloc[0]]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, row in zip(axes, examples):
    ax.imshow(render(row.page, row.bbox))
    ax.set_title(f"page {row.page}: {row.question}\nanswer: {row.answer}", fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## Scoring answers

The answers are short: a count, a part name, a size. A model that says "toward the rear" when the reference says "toward rear" has the content right, but a plain string comparison marks it wrong. The graded score is therefore the normalized match. Both strings are lowercased, × becomes x, punctuation and brackets are removed, a number written with a leading decimal point gets a leading zero (so .375 and 0.375 match), the articles a, an and the are dropped, and the words that are left must be the same. Exact match, where the two strings are equal once surrounding spaces are trimmed, is reported beside it. The difference between the two tells you how much of a score comes from wording rather than content.

In [ ]:
ARTICLES = {"a", "an", "the"}


def normalize(text):
    # Lowercase, x for ×, words and numbers only (decimals kept, .375 read as 0.375), articles dropped.
    words = re.findall(r"(?<![a-z0-9#.])\.\d+|[a-z0-9#]+(?:\.\d+)*", str(text).lower().replace("×", "x"))
    return " ".join("0" + w if w.startswith(".") else w for w in words if w not in ARTICLES)


def exact_match(pred, answer):
    return str(pred).strip() == str(answer).strip()


def normalized_match(pred, answer):
    return normalize(pred) == normalize(answer)


for pred, answer in [("toward the rear", "toward rear"), ("(1)", "1"), ("Terminal 20.", "20"), ("#6 x .375", "#6 X .375"), ("#6 x 0.375", "#6 x .375"), ("#6 x 375", "#6 x .375")]:
    print(f"{pred!r:20} {answer!r:14} exact {exact_match(pred, answer)!s:5}  normalized {normalized_match(pred, answer)}")

## The model

SmolVLM-256M-Instruct joins a small image encoder to the SmolLM2 language model through a connector that turns image patches into tokens. Each image is resized so that its longer side is 512 pixels and read as a single tile. The question goes in as a chat turn with the image placed before the text, and the answer is generated greedily, at most 24 tokens long.

`evaluate` runs the model on a set of pairs and returns one row per pair, with the prediction and both scores.

In [ ]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
processor.image_processor.size = {"longest_edge": IMAGE_EDGE}


def load_model(dtype=torch.float32):
    return AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=dtype).to(DEVICE)


def chat(question, answer=None):
    # One user turn (image, then question), and the reference answer as the assistant turn when training.
    turns = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": question + " Answer briefly."}]}]
    if answer is not None:
        turns.append({"role": "assistant", "content": [{"type": "text", "text": answer}]})
    return turns


@torch.no_grad()
def evaluate(model, df, max_new_tokens=24):
    model.eval()
    rows = []
    for r in df.itertuples():
        prompt = processor.apply_chat_template(chat(r.question), add_generation_prompt=True)
        x = processor(text=prompt, images=[render(r.page, r.bbox)], return_tensors="pt").to(DEVICE)
        x["pixel_values"] = x["pixel_values"].to(model.dtype)
        with torch.autocast("cuda", dtype=torch.float16):
            y = model.generate(**x, max_new_tokens=max_new_tokens, do_sample=False)
        pred = processor.batch_decode(y[:, x["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
        rows.append({"id": r.id, "kind": r.kind, "qtype": r.qtype, "question": r.question,
                     "answer": r.answer, "pred": pred})
    return score(pd.DataFrame(rows))


def score(out):
    out["exact"] = [exact_match(p, a) for p, a in zip(out.pred, out.answer)]
    out["normalized"] = [normalized_match(p, a) for p, a in zip(out.pred, out.answer)]
    return out


def fingerprint(configure):
    # Identifies a method's code: the source of its configure function and the plain values it reads, such as TARGETS.
    if configure is None:
        return None  # zero-shot trains nothing
    try:
        code = inspect.getsource(configure)
    except (OSError, TypeError):
        code = marshal.dumps(configure.__code__).hex()  # the compiled function when its source is not available
    values = {n: repr(v) for n in configure.__code__.co_names
              if isinstance(v := configure.__globals__.get(n), (str, int, float, list, tuple, dict))}
    return hashlib.sha256((code + json.dumps(values, sort_keys=True)).encode()).hexdigest()


def is_saved(name, code=None):
    # A saved run counts only when it was made with the current SETTINGS and the same method code.
    meta = RUNS / f"{name}.json"
    if not ((RUNS / f"{name}.csv").exists() and meta.exists()):
        return False
    saved = json.loads(meta.read_text())
    return saved.get("settings") == SETTINGS and saved.get("code") == code


def load_run(name):
    # The saved predictions, scored again with the current metric.
    return score(pd.read_csv(RUNS / f"{name}.csv", dtype={"answer": str, "pred": str}, keep_default_na=False))

## Part 1: the zero-shot baseline (20 points)

The base model answers every test question before any training. Its results are saved as `runs/zeroshot.csv` so that Part 3 can compare it with the fine-tuned models.

In [ ]:
ZEROSHOT = RUNS / "zeroshot.csv"
if is_saved("zeroshot"):
    zeroshot = load_run("zeroshot")
else:
    base = load_model(torch.float16)
    start = time.time()
    zeroshot = evaluate(base, test_df)
    zeroshot.to_csv(ZEROSHOT, index=False)
    (RUNS / "zeroshot.json").write_text(json.dumps({"method": "zeroshot", "trainable": 0, "peak_gib": 0.0,
                                                     "train_min": 0.0, "checkpoint_mb": 0.0, "losses": [],
                                                     "settings": SETTINGS, "code": None}))
    print(f"{len(zeroshot)} test pairs in {(time.time() - start) / 60:.1f} min")
    del base
    torch.cuda.empty_cache()
print(f"normalized match {zeroshot.normalized.mean():.3f}   exact match {zeroshot.exact.mean():.3f}")
zeroshot.sample(min(8, len(zeroshot)), random_state=1)[["qtype", "question", "answer", "pred"]]

Write `by_qtype`, which breaks a set of results down by question type, and `failure_kind`, which sorts each failed prediction into one of three kinds. A prediction that contains the right answer and more around it is a different failure from one that gives the wrong number, and the two call for different fixes.

In [ ]:
def by_qtype(results):
    """Normalized and exact match per question type, with the number of test pairs.

    Return a DataFrame indexed by qtype with the columns "normalized", "exact" and "n",
    sorted by "normalized" from highest to lowest.
    """
    raise NotImplementedError("Part 1: write by_qtype")


def failure_kind(row):
    """Why one prediction failed the normalized match.

    Return "verbose" when the normalized answer appears inside the normalized prediction,
    "wrong number" when the answer is a whole number and the prediction does not contain that number,
    and "other" for every other failure.
    """
    raise NotImplementedError("Part 1: write failure_kind")

In [ ]:
try:
    display(by_qtype(zeroshot))
    failed = zeroshot[~zeroshot.normalized]
    display(failed.apply(failure_kind, axis=1).value_counts().rename("failed pairs"))
except NotImplementedError as e:
    print(e)

### 1. What the base model can and cannot do

Which question types does the base model already answer, and which does it fail? Use the per-type table and the failure kinds, and quote three predictions that show the pattern.

**Answer:** _Write your answer here._

## Part 2: fine-tuning (50 points)

Every method trains on the same objective: the cross-entropy of the reference answer's tokens, given the image and the question. The prompt tokens are masked out of the loss, so the model learns to produce the answer instead of repeating the question. One pair is one forward pass, and the gradients of four pairs are added up before each optimizer step. The weights are stored in 32-bit floats while the forward pass runs in 16-bit floats, and the loss is scaled up before the backward pass so that small gradients do not round to zero in 16 bits.

`run_method(name, configure)` loads a fresh base model, passes it to your `configure` function, trains whatever `configure` left trainable, and scores the result on the test split. It records the trainable parameter count, peak GPU memory, training time and the size of the saved checkpoint in `runs/<name>.json`, and the per-pair results in `runs/<name>.csv`. A method whose files already exist is loaded instead of retrained, but only if they were made with the same settings and the same `configure` function. Change `TRAIN_LIMIT`, `TEST_LIMIT`, `EPOCHS` or another setting and the method is trained again. Fix a bug in `configure`, or in a value it reads such as `TARGETS`, and it is trained again too. If `configure` calls a helper function of yours and you change only the helper, pass `retrain=True` to train it again.

In [ ]:
def train(model, df, epochs=EPOCHS, lr=LR, accum=ACCUM):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr)
    scaler = torch.amp.GradScaler()
    model.train()
    losses = []
    for epoch in range(epochs):
        order = df.sample(frac=1, random_state=epoch)
        for step, r in enumerate(order.itertuples()):
            image = render(r.page, r.bbox)
            full = processor.apply_chat_template(chat(r.question, r.answer))
            prompt = processor.apply_chat_template(chat(r.question), add_generation_prompt=True)
            x = processor(text=full, images=[image], return_tensors="pt").to(DEVICE)
            n_prompt = processor(text=prompt, images=[image], return_tensors="pt")["input_ids"].shape[1]
            labels = x["input_ids"].clone()
            labels[:, :n_prompt] = -100  # the loss covers the answer tokens only
            with torch.autocast("cuda", dtype=torch.float16):
                loss = model(**x, labels=labels).loss
            scaler.scale(loss / accum).backward()
            if (step + 1) % accum == 0 or step + 1 == len(order):
                scaler.step(opt)
                scaler.update()
                opt.zero_grad()
            losses.append(loss.item())
            if step % 200 == 0:
                print(f"epoch {epoch} step {step} loss {loss.item():.3f}", flush=True)
    return losses


def run_method(name, configure, retrain=False):
    code = fingerprint(configure)
    if is_saved(name, code) and not retrain:
        print(f"{name}: loaded from {RUNS}")
        return load_run(name)
    torch.manual_seed(0)
    model = configure(load_model())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"{name}: {trainable:,} trainable parameters")
    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    losses = train(model, train_df)
    minutes = (time.time() - start) / 60
    peak = torch.cuda.max_memory_allocated() / 2**30
    checkpoint = RUNS / name
    model.save_pretrained(checkpoint)
    size = sum(f.stat().st_size for f in checkpoint.rglob("*") if f.is_file()) / 2**20
    results = evaluate(model, test_df)
    results.to_csv(RUNS / f"{name}.csv", index=False)
    (RUNS / f"{name}.json").write_text(json.dumps({"method": name, "trainable": trainable, "peak_gib": peak,
                                                   "train_min": minutes, "checkpoint_mb": size, "losses": losses,
                                                   "settings": SETTINGS, "code": code}))
    print(f"{name}: normalized match {results.normalized.mean():.3f}, exact match {results.exact.mean():.3f}, "
          f"{minutes:.1f} min, peak {peak:.2f} GiB")
    del model
    torch.cuda.empty_cache()
    return results

Write the three `configure` functions. Each takes the base model and returns a model in which only the method's parameters are trainable. The parameter counts in the docstrings let you check your setup before you spend time training: `run_method` prints the count as soon as `configure` returns.

In [ ]:
# LoRA and DoRA wrap these projections, in the vision encoder and in the language model.
TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]


def lora(model):
    """LoRA of rank 8 on every module named in TARGETS: lora_alpha=8, lora_dropout=0.1, Gaussian init.

    Use peft.LoraConfig and peft.get_peft_model. With these settings 2,884,608 parameters train.
    """
    raise NotImplementedError("Part 2: write lora")


def dora(model):
    """The same adapters as lora(), with the weight-decomposed (DoRA) update switched on.

    With these settings 3,067,776 parameters train.
    """
    raise NotImplementedError("Part 2: write dora")


def ln_tuning(model):
    """Train only the language model's normalization layers: the two in each decoder layer and the final one.

    Use peft.LNTuningConfig. Print the names from model.named_modules() to find the modules to target.
    With these settings 35,136 parameters train.
    """
    raise NotImplementedError("Part 2: write ln_tuning")

### LoRA

In [ ]:
try:
    run_method("lora", lora)
except NotImplementedError as e:
    print(e)

### DoRA

In [ ]:
try:
    run_method("dora", dora)
except NotImplementedError as e:
    print(e)

### LayerNorm tuning

In [ ]:
try:
    run_method("ln_tuning", ln_tuning)
except NotImplementedError as e:
    print(e)

## Part 3: comparing the methods (30 points)

`compare` collects every finished run from `runs/` into one table: the cost of each method next to both scores. A run counts as finished only if it was made with the current settings and the current version of its `configure` function, so an older run of code you have since fixed is left out. The plots below show the training loss of each method and its normalized match per question type.

In [ ]:
METHODS = {"zeroshot": None, "lora": "lora", "dora": "dora", "ln_tuning": "ln_tuning"}  # the name of each configure function


def finished():
    # Only runs made with the current settings and each configure function as it is defined right now.
    return [m for m, fn in METHODS.items() if is_saved(m, fingerprint(globals()[fn] if fn else None))]


def compare():
    rows = []
    for m in finished():
        cost = json.loads((RUNS / f"{m}.json").read_text())
        res = load_run(m)
        rows.append({"method": m, "trainable": cost["trainable"], "peak GiB": round(cost["peak_gib"], 2),
                     "train min": round(cost["train_min"], 1), "checkpoint MB": round(cost["checkpoint_mb"], 1),
                     "normalized": round(res.normalized.mean(), 3), "exact": round(res.exact.mean(), 3)})
    return pd.DataFrame(rows).set_index("method")


compare()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for m in finished():
    losses = pd.Series(json.loads((RUNS / f"{m}.json").read_text())["losses"])
    if len(losses):
        ax.plot(losses.rolling(50).mean(), label=m)
ax.set_xlabel("training pair")
ax.set_ylabel("loss, mean of the last 50 pairs")
ax.set_yscale("log")
ax.legend()
plt.show()

In [ ]:
per_type = pd.DataFrame({m: load_run(m).groupby("qtype").normalized.mean() for m in finished()})
ax = per_type.plot.bar(figsize=(12, 4), width=0.8)
ax.set_ylabel("normalized match")
ax.set_xlabel("question type")
ax.legend(title="method")
plt.tight_layout()
plt.show()

### 2. Choosing a method

Rank the three fine-tuning methods by normalized match and by cost: trainable parameters, peak memory, training time and checkpoint size. Which would you deploy, and why? You trained each method once, so say whether the gap between the two closest methods could come from the seed alone, and what you would run to find out.

**Answer:** _Write your answer here._

### 3. Failure cases

Take your best method and pick five test pairs it gets wrong under the normalized match. Show the image, the question, the reference and the prediction for each in the cell below. For each one, say whether the reference, the normalization or the model is at fault.

**Answer:** _Write your answer here._

In [ ]:
# Five failure cases of your best method

### 4. Leakage

Every step image shows its own printed instruction, and the test pairs come from page blocks that training never sees, with neighboring pages and shared photos removed.

1. Explain why a random split of the 1,493 pairs would have given a higher test score than this split, and why that score would mislead you.
2. Name one way information about the test pairs can still reach training under this split, and describe how you would measure its effect on the score.

**Answer:** _Write your answer here._

## Submitting

Commit this file with every cell output saved, together with the CSV and JSON files in `runs/`, under `assignments/assignment-3` in your clone of the class repository. Leave out the checkpoint folders in `runs/`. Submit the GitHub URL of that folder, not the repository root.